# **IMPORTS ET JETON**

In [11]:
import os
import requests
from dotenv import load_dotenv

import json

In [2]:

load_dotenv()
headers = {"Authorization": f"Bearer {os.getenv('TMDB_TOKEN')}"}

# **RECUPERER LA LISTE DES FILMS AVEC "discover"**

### Endpoint utilisé : `/discover/movie`
Cet endpoint fonctionne comme une **recherche avancée** : on lui donne des critères (dates, nombre de votes, tri…)
et il renvoie les films correspondants, **par pages de 20**.

📖 Documentation officielle : https://developer.themoviedb.org/reference/discover-movie


In [3]:
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "language": "fr-FR",
    "primary_release_date.gte": "2000-01-01",
    "primary_release_date.lte": "2025-12-31",
    "vote_count.gte": 200,
    "sort_by": "vote_count.desc",
    "page": 1,
}
reponse = requests.get(url, headers=headers, params=params)
data = reponse.json()

print("Code HTTP :", reponse.status_code)
print("Films trouvés :", data["total_results"])
print("Nombre de pages :", data["total_pages"])
print("Films par page :", len(data["results"]))

Code HTTP : 200
Films trouvés : 10810
Nombre de pages : 541
Films par page : 20


In [4]:
for f in data["results"][:10]:
    print(f["release_date"], "|", f["title"], "|", f["vote_count"], "votes")

2014-11-04 | Interstellar | 41998 votes
2010-07-15 | Inception | 40808 votes
2012-04-25 | Avengers | 40452 votes
2008-07-16 | The Dark Knight : Le Chevalier noir | 37466 votes
2009-12-16 | Avatar | 35071 votes
2016-02-09 | Deadpool | 33670 votes
2018-04-25 | Avengers : Infinity War | 33466 votes
2014-07-30 | Les Gardiens de la Galaxie | 30707 votes
2001-11-16 | Harry Potter à l'école des sorciers | 30677 votes
2019-04-24 | Avengers : Endgame | 29489 votes


In [5]:
for minimum in [50, 100, 200, 500, 1000]:
    params["vote_count.gte"] = minimum
    data = requests.get(url, headers=headers, params=params).json()
    print(f"Au moins {minimum} votes → {data['total_results']} films")

Au moins 50 votes → 20001 films
Au moins 100 votes → 16473 films
Au moins 200 votes → 10810 films
Au moins 500 votes → 6015 films
Au moins 1000 votes → 3800 films


In [8]:
params["vote_count.gte"] = 200
params["page"] = 501   # on demande une page au-delà de 500

reponse = requests.get(url, headers=headers, params=params)
print("Code HTTP :", reponse.status_code)
print(reponse.json())

Code HTTP : 400
{'success': False, 'status_code': 22, 'status_message': 'Invalid page: Pages start at 1 and max at 500. They are expected to be an integer.'}


Suite à cette micro analyse des données, on conservera les films qui ont "au moins" 500 votes soit 6015 films.

Le seuil de 500 est jugé par le fait que si on récupère un volume de vote plus bas, les films seraient effectivement plus nombreux, mais avec des informations manquantes (budget, recettes etc...). 
De plus un seuil de 200 annonce 541 pages, ce qui dépasse la limite imposé par l'API ("Pages start at 1 and max 500").
A l'inverse, les votes à 1000 ne concerne que 3800 films, ce qui parait léger pour le Machine Learning. 

In [9]:
import time  # pour faire une petite pause entre les requêtes et ne pas surcharger l'API


In [10]:

params["vote_count.gte"] = 500   # seuil retenu

ids_films = []      # liste qui contiendra tous les identifiants
page = 1            # on commence à la première page
total_pages = 1     # valeur provisoire, mise à jour après la première requête

while page <= total_pages:
    params["page"] = page
    data = requests.get(url, headers=headers, params=params).json()

    # on ajoute l'id de chaque film de la page à la liste
    for film in data["results"]:
        ids_films.append(film["id"])

    total_pages = data["total_pages"]   # nombre réel de pages, donné par l'API
    page += 1                           # page suivante
    time.sleep(0.05)                    # pause de 50 ms pour ne pas surcharger l'API

print("Pages parcourues :", total_pages)
print("Identifiants récupérés :", len(ids_films))
print("Identifiants uniques :", len(set(ids_films)))

Pages parcourues : 301
Identifiants récupérés : 6015
Identifiants uniques : 6015


*On sauvegarde la liste des ingrédients dans un fichier JSON*

In [12]:
#  dossier "data" 
os.makedirs("../data", exist_ok=True)

with open("../data/ids_films.json", "w") as fichier:
    json.dump(ids_films, fichier)

print(len(ids_films), "identifiants enregistrés dans data/ids_films.json")

6015 identifiants enregistrés dans data/ids_films.json
